In [ ]:
import sys
import os
if not '../..' in sys.path:
    sys.path.append('../..')
import basico
import pandas
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
fname = "../Chass_simulation_data.xlsx"
data = pandas.read_excel(fname, sheet_name=1)

In [20]:
with open("../chass_testbatch.txt", "r") as f:
    testbatch_str = f.read().split()
    testbatch = set([int(el) for el in testbatch_str])
prev_val = 0
batches_lengths = []
for val in data["Experiment"]:
    if val != prev_val:
        batches_lengths.append(1)
        prev_val = val
    else:
        batches_lengths[-1] += 1
not_testbatch = set(range(len(batches_lengths))) - testbatch
not_testbatch_exp_labels = [el + 1 for el in not_testbatch]

In [21]:
maxes = dict()
mins = dict()
concentrations_labels = ['adp', 'asa', 'asp', 'aspp', 'atp', 'hs', 'hsp', 'nadp', 'nadph', 'phos', 'thr']
for label in (['time'] + concentrations_labels):
    max_val = max(data[data["Experiment"].isin(not_testbatch_exp_labels)][label])
    min_val = min(data[data["Experiment"].isin(not_testbatch_exp_labels)][label])
    label_scaled = f"{label}_scaled"
    maxes[label] = max_val
    mins[label] = min_val
    data[label_scaled] = (data[label] - min_val) / (max_val - min_val)
maxes_for_columns = torch.tensor([[maxes[label] for label in concentrations_labels]])
mins_for_columns = torch.tensor([[mins[label] for label in concentrations_labels]])

In [22]:
prev_val = 0
batches_lengths = []
for val in data["Experiment"]:
    if val != prev_val:
        batches_lengths.append(1)
        prev_val = val
    else:
        batches_lengths[-1] += 1
batches = []
for i, length in enumerate(batches_lengths):
    batches.append(data.iloc[i*length : i*length + length].reset_index(drop=True))
length = batches_lengths[0]

#scale inputs
changing_vars = ['asp', 'atp']
first_of_each_batch = data[changing_vars].iloc[::length]
mins_Xs = first_of_each_batch.min()
maxes_Xs = first_of_each_batch.max()
scaled_initial_conditions = (first_of_each_batch - mins_Xs)/(maxes_Xs-mins_Xs)

def make_constant_input_label(label):
    return f"{label}0"

mins_Xs_dict = {make_constant_input_label(label): minX for label, minX in zip(changing_vars, mins_Xs)}
maxes_Xs_dict = {make_constant_input_label(label): maxX for label, maxX in zip(changing_vars, maxes_Xs)}

Xs_ys = []
for i, batch in enumerate(batches):
    scaled_initials_for_this_batch = pandas.concat([scaled_initial_conditions.iloc[i:i+1]]*length, ignore_index=True)
    X = pandas.concat([batch[["time_scaled"]], scaled_initials_for_this_batch], axis=1)
    y = batch[[f"{label}_scaled" for label in concentrations_labels]]
    Xs_ys.append((X, y))

In [23]:
X_all = torch.stack([torch.Tensor(batch[0].values) for batch in Xs_ys])
y_all = torch.stack([torch.Tensor(batch[1].values) for batch in Xs_ys])

all_data = TensorDataset(X_all, y_all)
    
all_dataloader = DataLoader(all_data, batch_size=len(all_data))

In [24]:
INPUTS = 3
OUTPUTS = 11

class ChassFfnnRegressor(nn.Module):
    def __init__(self, inputs, layers, outputs, activation):
        all_layers = [inputs] + layers + [outputs]
        super().__init__()
        self.linears = nn.ModuleList([nn.Linear(all_layers[i], all_layers[i + 1]) for i in range(len(all_layers) - 1)])
        if activation == "tanh":
            self.act = nn.Tanh()
        elif activation == "relu":
            self.act = nn.ReLU()
    
    def forward(self, x):
        for hidden in self.linears[:-1]:
            x = self.act(hidden(x))
        x = self.linears[-1](x)
        return x

In [25]:
NUM_HIDDEN = 1

class ChassRnnRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, activation):
        super(ChassRnnRegressor, self).__init__()
        
        # Number of hidden dimensions
        self.hidden_dim = hidden_dim
        
        # Number of hidden layers
        self.layer_dim = layer_dim
        
        # RNN
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, batch_first=True, nonlinearity=activation)
        
        # Readout layer
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim)
            
        # One time step
        out, hn = self.rnn(x, h0)
        out = self.fc(out) 
        return out

In [26]:
num_nodes = {
    "ffnn":[[5, 5], [10, 10], [16, 16], [16, 16, 16]],
    "rnn": [10, 15, 18, 21]
}
activations = ["tanh", "relu"]

In [27]:
def load_model(model_name, path_to_folder, model_type, activation, layers):
    if model_type == "ffnn":
        nn_model = ChassFfnnRegressor(INPUTS, layers, OUTPUTS, activation)
    elif model_type == "rnn":
        nn_model = ChassRnnRegressor(INPUTS, layers, NUM_HIDDEN, OUTPUTS, activation)
    nn_model.load_state_dict(torch.load(os.path.join(path_to_folder, model_name), weights_only=True, map_location=torch.device('cpu')))
    nn_model.eval()
    return nn_model

In [28]:
def get_nodes_str(nodes):
    if type(nodes) is list:
        return "_".join([str(node) for node in nodes])
    else:
        return str(nodes)

In [29]:
def adjust_model_to_batch(batch_number, labels):
    this_X = batches[batch_number]
    for label in labels:
        value = this_X[label].iloc[0] 
        basico.set_parameters(make_constant_input_label(label), initial_value=value)

In [30]:
def get_sds(maxes_for_columns):
    sds = [value * 0.1 for value in maxes_for_columns]
    return sds

In [31]:
def calculate_basico_prediction(document_name, path_to_folder):
    duration = len(Xs_ys[0][1]) - 1
    step_number = duration

    output_labels = concentrations_labels
    
    basico.load_model(os.path.join(path_to_folder, document_name))

    copasi_preds = []
    for batch_number in range(len(batches)):
        adjust_model_to_batch(batch_number, changing_vars)
        df = basico.run_time_course(step_number=step_number, duration=duration)
        copasi_pred = torch.tensor(df[output_labels].values)
        copasi_preds.append(copasi_pred)
    copasi_preds = torch.stack(copasi_preds)
    return copasi_preds

In [32]:
def calculate_ann_predictions(nn_model, X_all):
    y_pred = nn_model(X_all)
    y_pred_scaled = y_pred * (maxes_for_columns - mins_for_columns) + mins_for_columns
    return y_pred_scaled

In [33]:
model_encodings = {
    "ffnn": ["big_assignment_rules", "layers_as_compartments", "nodes_as_parameters", "nodes_as_species"],
    "ernn": ["big_event_assignments", "layers_as_compartments", "nodes_as_parameters", "nodes_as_species"]
}
def get_sbml_filenames(model_type, activation, layers):
    if model_type == "rnn":
        model_type = "ernn"
    return [f"chass_{model_type}_{activation}_{get_nodes_str(layers)}_{encoding}_sbml.xml" for encoding in model_encodings[model_type]]

In [34]:
# load models
path_to_folder = "../chass_best_models"
path_to_sbml_folder = "../chass_different_encodings"
with torch.no_grad():
    for model_type, nodes_s in num_nodes.items():
        for activation in activations:
            for layers in nodes_s:
                model_name = f"Chass_{model_type}_{activation}_{get_nodes_str(layers)}_best"
                nn_model = load_model(model_name, path_to_folder, model_type, activation, layers)
                pred = calculate_ann_predictions(nn_model, X_all)
                sbml_names = get_sbml_filenames(model_type, activation, layers)
                for sbml_name in sbml_names:
                    copasi_preds = calculate_basico_prediction(sbml_name, path_to_sbml_folder)
                    mse = torch.nn.functional.mse_loss(pred, copasi_preds)
                    print(sbml_name, mse.item())

chass_ffnn_tanh_5_5_big_assignment_rules_sbml.xml 5.70946889854568e-14
chass_ffnn_tanh_5_5_layers_as_compartments_sbml.xml 5.70946889854568e-14
chass_ffnn_tanh_5_5_nodes_as_parameters_sbml.xml 5.70946889854568e-14
chass_ffnn_tanh_5_5_nodes_as_species_sbml.xml 5.70946889854568e-14
chass_ffnn_tanh_10_10_big_assignment_rules_sbml.xml 5.241036182100099e-14
chass_ffnn_tanh_10_10_layers_as_compartments_sbml.xml 5.241036182100099e-14
chass_ffnn_tanh_10_10_nodes_as_parameters_sbml.xml 5.241036182100099e-14
chass_ffnn_tanh_10_10_nodes_as_species_sbml.xml 5.241036182100099e-14
chass_ffnn_tanh_16_16_big_assignment_rules_sbml.xml 5.973719318950264e-14
chass_ffnn_tanh_16_16_layers_as_compartments_sbml.xml 5.973719318950264e-14
chass_ffnn_tanh_16_16_nodes_as_parameters_sbml.xml 5.973719318950264e-14
chass_ffnn_tanh_16_16_nodes_as_species_sbml.xml 5.973719318950264e-14
chass_ffnn_tanh_16_16_16_big_assignment_rules_sbml.xml 8.725403125646876e-14
chass_ffnn_tanh_16_16_16_layers_as_compartments_sbml.xml